In [17]:
!apt-get update -qq
!apt-get install -y swig > /dev/null
!pip install -q "gymnasium[box2d]" moviepy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [18]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
import cv2, random, os, glob, time, base64, datetime
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

random.seed(0); np.random.seed(0); torch.manual_seed(0)

Using device: cuda


In [19]:
class CarRacingWrapper(gym.Wrapper):
    def __init__(self, env, k=4, action_repeat=4, patience=50, skip_start=50):
        super().__init__(env)
        self.k, self.action_repeat = k, action_repeat
        self.patience, self.skip_start = patience, skip_start
        self.frames = deque(maxlen=k)
        self.no_progress = 0
        self.observation_space = gym.spaces.Box(0, 255, (k, 96, 96), np.uint8)

    @staticmethod
    def _gray(obs):
        return cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)

    def _off_track(self):
        # True when none of the 4 wheels touches a road tile
        wheels = self.env.unwrapped.car.wheels
        return all(len(w.tiles) == 0 for w in wheels)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.skip_start):  # skip the zoom-in intro
            obs, *_ = self.env.step(np.zeros(3, dtype=np.float32))
        g = self._gray(obs)
        self.frames.clear()
        for _ in range(self.k):
            self.frames.append(g)
        self.no_progress = 0
        return np.stack(self.frames), info

    def step(self, action):
        raw, terminated, truncated = 0.0, False, False
        for _ in range(self.action_repeat):
            obs, r, terminated, truncated, info = self.env.step(action)
            raw += r
            if terminated or truncated:
                break

        shaped = raw
        if self._off_track():
            shaped -= 1.0                       # penalty for driving on grass
        self.no_progress = 0 if raw > 0 else self.no_progress + 1

        reward = float(np.clip(shaped * 0.25, -2.0, 2.0))
        if self.no_progress >= self.patience and not (terminated or truncated):
            terminated = True                   # stuck / lost: end the episode early
            reward -= 1.0

        self.frames.append(self._gray(obs))
        info = dict(info)
        info["raw_reward"] = raw                # real game score, used for logging
        return np.stack(self.frames), reward, terminated, truncated, info

In [20]:
def _init(m, gain=np.sqrt(2)):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.orthogonal_(m.weight, gain)
        nn.init.zeros_(m.bias)

class CNNPolicy(nn.Module):
    def __init__(self, obs_shape, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(obs_shape[0], 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
        )
        conv_out = self._get_conv_out(obs_shape)
        self.actor = nn.Sequential(nn.Linear(conv_out, 256), nn.ReLU(), nn.Linear(256, n_actions))
        self.critic = nn.Sequential(nn.Linear(conv_out, 256), nn.ReLU(), nn.Linear(256, 1))
        self.apply(_init)
        nn.init.orthogonal_(self.actor[-1].weight, 0.01)
        nn.init.orthogonal_(self.critic[-1].weight, 1.0)

    def _get_conv_out(self, shape):
        return int(np.prod(self.conv(torch.zeros(1, *shape)).shape))

    def forward(self, x):
        h = self.conv(x / 255.0).flatten(start_dim=1)
        return self.actor(h), self.critic(h)

    def act(self, x):
        logits, value = self.forward(x)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value

    def evaluate(self, x, actions):
        logits, value = self.forward(x)
        dist = torch.distributions.Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), value.squeeze(-1)

In [21]:
ACTION_MAP = {
    0: np.array([ 0.0, 1.0, 0.0], dtype=np.float32),  # gas
    1: np.array([-1.0, 0.5, 0.0], dtype=np.float32),  # left + gas
    2: np.array([ 1.0, 0.5, 0.0], dtype=np.float32),  # right + gas
    3: np.array([ 0.0, 0.0, 0.5], dtype=np.float32),  # brake
    4: np.array([ 0.0, 0.0, 0.0], dtype=np.float32),  # coast
}
N_ACTIONS = len(ACTION_MAP)

TOTAL_STEPS   = 300_000   # agent steps (each = 4 game frames). Raise for better driving.
ROLLOUT_STEPS = 2048
PPO_EPOCHS    = 4
MINIBATCH     = 256
LR            = 2.5e-4
GAMMA, LAM    = 0.99, 0.95
CLIP_EPS      = 0.2
ENT_COEF      = 0.01
VF_COEF       = 0.5
PATIENCE      = 50        # agent steps without progress before an episode is cut

In [22]:
env = CarRacingWrapper(gym.make("CarRacing-v3", render_mode="rgb_array"), patience=PATIENCE)
policy = CNNPolicy(env.observation_space.shape, N_ACTIONS).to(device)
optimizer = optim.Adam(policy.parameters(), lr=LR, eps=1e-5)

num_updates = TOTAL_STEPS // ROLLOUT_STEPS
recent = deque(maxlen=10)
best_score, ep_raw, start = -1e9, 0.0, time.time()
obs, _ = env.reset(seed=0)

for update in range(1, num_updates + 1):
    # linear learning-rate decay
    for g in optimizer.param_groups:
        g["lr"] = LR * (1 - (update - 1) / num_updates)

    T = ROLLOUT_STEPS
    b_obs = torch.zeros((T, *env.observation_space.shape), dtype=torch.uint8, device=device)
    b_act = torch.zeros(T, dtype=torch.long, device=device)
    b_logp = torch.zeros(T, device=device)
    b_val = torch.zeros(T, device=device)
    b_rew = torch.zeros(T, device=device)
    b_done = torch.zeros(T, device=device)

    # ---- Rollout ----
    for t in range(T):
        obs_t = torch.as_tensor(obs, device=device)
        b_obs[t] = obs_t
        with torch.no_grad():
            action, logp, _, value = policy.act(obs_t.float().unsqueeze(0))
        a = action.item()
        next_obs, reward, terminated, truncated, info = env.step(ACTION_MAP[a])
        done = terminated or truncated

        b_act[t], b_logp[t] = a, logp.item()
        b_val[t], b_rew[t], b_done[t] = value.item(), reward, float(done)
        ep_raw += info["raw_reward"]

        if done:
            recent.append(ep_raw)
            ep_raw = 0.0
            obs, _ = env.reset()
        else:
            obs = next_obs

    # ---- GAE ----
    with torch.no_grad():
        last_val = policy(torch.as_tensor(obs, device=device).float().unsqueeze(0))[1].item()
    adv = torch.zeros(T, device=device)
    lastgae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - b_done[t].item()
        next_val = last_val if t == T - 1 else b_val[t + 1].item()
        delta = b_rew[t].item() + GAMMA * next_val * nonterminal - b_val[t].item()
        lastgae = delta + GAMMA * LAM * nonterminal * lastgae
        adv[t] = lastgae
    ret = adv + b_val
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    # ---- PPO update ----
    for _ in range(PPO_EPOCHS):
        perm = torch.randperm(T, device=device)
        for s in range(0, T, MINIBATCH):
            idx = perm[s:s + MINIBATCH]
            new_logp, entropy, new_val = policy.evaluate(b_obs[idx].float(), b_act[idx])
            ratio = torch.exp(new_logp - b_logp[idx])
            surr1 = ratio * adv[idx]
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv[idx]
            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = F.mse_loss(new_val, ret[idx])
            loss = actor_loss + VF_COEF * critic_loss - ENT_COEF * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()

    # ---- Logging / checkpoint ----
    mean_score = np.mean(recent) if recent else float("nan")
    print(f"Update {update}/{num_updates} | steps {update * T} | "
          f"avg score (last {len(recent)} eps): {mean_score:.1f} | {time.time() - start:.0f}s")
    if len(recent) >= 5 and mean_score > best_score:
        best_score = mean_score
        torch.save(policy.state_dict(), "best_policy.pt")

env.close()
torch.save(policy.state_dict(), "final_policy.pt")

Update 1/146 | steps 2048 | avg score (last 10 eps): 2.8 | 104s
Update 2/146 | steps 4096 | avg score (last 10 eps): 8.7 | 197s
Update 3/146 | steps 6144 | avg score (last 10 eps): 7.3 | 287s
Update 4/146 | steps 8192 | avg score (last 10 eps): 2.4 | 375s
Update 5/146 | steps 10240 | avg score (last 10 eps): 15.6 | 461s
Update 6/146 | steps 12288 | avg score (last 10 eps): 0.5 | 549s
Update 7/146 | steps 14336 | avg score (last 10 eps): 4.1 | 637s
Update 8/146 | steps 16384 | avg score (last 10 eps): 12.8 | 724s
Update 9/146 | steps 18432 | avg score (last 10 eps): 27.5 | 811s
Update 10/146 | steps 20480 | avg score (last 10 eps): 23.9 | 897s
Update 11/146 | steps 22528 | avg score (last 10 eps): 46.7 | 985s
Update 12/146 | steps 24576 | avg score (last 10 eps): 48.0 | 1073s
Update 13/146 | steps 26624 | avg score (last 10 eps): 71.6 | 1160s
Update 14/146 | steps 28672 | avg score (last 10 eps): 76.3 | 1247s
Update 15/146 | steps 30720 | avg score (last 10 eps): 52.0 | 1338s
Update 16/

KeyboardInterrupt: 

In [24]:
if os.path.exists("best_policy.pt"):
    policy.load_state_dict(torch.load("best_policy.pt", map_location=device))
    print("Loaded best checkpoint, score:", round(best_score, 1))
policy.eval()

DETERMINISTIC = False   # False = sample actions instead of always taking the most likely one
video_folder = f"videos_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

eval_env = gym.make("CarRacing-v3", render_mode="rgb_array", max_episode_steps=2000)
eval_env = RecordVideo(eval_env, video_folder=video_folder,
                       episode_trigger=lambda e: True, name_prefix="final_run")
eval_env = CarRacingWrapper(eval_env, patience=10**9)   # no early cut-off for the demo

obs, _ = eval_env.reset(seed=123)
done, score = False, 0.0
while not done:
    with torch.no_grad():
        logits, _ = policy(torch.as_tensor(obs, device=device).float().unsqueeze(0))
        a = logits.argmax(-1).item() if DETERMINISTIC else torch.distributions.Categorical(logits=logits).sample().item()
    obs, _, terminated, truncated, info = eval_env.step(ACTION_MAP[a])
    score += info["raw_reward"]
    done = terminated or truncated
eval_env.close()
print(f"Final run score: {score:.1f}")

for path in glob.glob(f"{video_folder}/*.mp4"):
    data = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

Loaded best checkpoint, score: 404.7
Final run score: 152.8


In [25]:
policy.load_state_dict(torch.load("best_policy.pt", map_location=device))
policy.eval()

def run_eval(n_eps, deterministic, seed0=1000):
    scores = []
    for i in range(n_eps):
        e = CarRacingWrapper(gym.make("CarRacing-v3", render_mode="rgb_array",
                                      max_episode_steps=2000), patience=10**9)
        obs, _ = e.reset(seed=seed0 + i)
        done, score = False, 0.0
        while not done:
            with torch.no_grad():
                logits, _ = policy(torch.as_tensor(obs, device=device).float().unsqueeze(0))
                if deterministic:
                    a = logits.argmax(-1).item()
                else:
                    a = torch.distributions.Categorical(logits=logits).sample().item()
            obs, _, term, trunc, info = e.step(ACTION_MAP[a])
            score += info["raw_reward"]
            done = term or trunc
        e.close()
        scores.append(score)
    return scores

for det in (True, False):
    s = run_eval(5, det)
    print("deterministic" if det else "sampled", [round(x) for x in s], "mean", round(np.mean(s), 1))

deterministic [-28, -105, -90, -105, -6] mean -66.6
sampled [16, 186, 347, 762, 292] mean 320.3


In [26]:
s = run_eval(20, deterministic=False, seed0=2000)
print([round(x) for x in s])
print("mean", round(np.mean(s), 1), "| std", round(np.std(s), 1),
      "| min", round(min(s)), "| max", round(max(s)))

[489, 514, 231, 233, 98, 777, 326, 443, 505, 378, 173, 169, 468, 779, 112, 667, 758, 788, 602, -72]
mean 421.9 | std 251.9 | min -72 | max 788
